<p align="center">
  <img src="https://www.copernicuslac-chile.eu/wp-content/uploads/2024/05/logo_copernicuslac_chile.svg" alt="Copernicus LAC Chile" width="1600"/>
</p>

# 00 — Verificación del entorno

Taller **CopernicusLAC Chile — IDE Uruguay**

Ejecuta este notebook al inicio del taller para comprobar Python, los paquetes principales usados por los notebooks y la estructura de los datos versionados del repositorio.

Una dependencia o una operación que falla se informa como **ERROR**. Si no hay un archivo opcional para probar, se informa como **AVISO** y eso por sí solo no impide usar el entorno.

## 1. Python y kernel

El entorno definido para el taller es `2026coplac`, con Python 3.11.

In [6]:
import importlib
import importlib.util
import platform
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
print("Sistema operativo:", platform.system(), platform.release())
print("Entorno esperado: 2026coplac (Python 3.11)")

Python: 3.11.17
Ejecutable: c:\Users\Paula_Copelli\miniconda3\envs\2026coplac\python.exe
Sistema operativo: Windows 10
Entorno esperado: 2026coplac (Python 3.11)


## 2. Dependencias

Se comprueban las librerías principales que usan los notebooks del taller. El resto de las dependencias de `environment.yml` no se prueba aquí.

In [7]:
DEPENDENCIAS = [
    ("Base", "numpy", "numpy"),
    ("Base", "pandas", "pandas"),
    ("Visualización", "matplotlib", "matplotlib"),
    ("Geoespacial", "geopandas", "geopandas"),
    ("Geoespacial", "shapely", "shapely"),
    ("Geoespacial", "rasterio", "rasterio"),
    ("Jupyter", "IPython", "ipython"),
    ("Jupyter", "ipywidgets", "ipywidgets"),
    ("Jupyter", "ipykernel", "ipykernel"),
]

resultados_dependencias = []
modulos = {}

for grupo, nombre_import, nombre_paquete in DEPENDENCIAS:
    disponible = importlib.util.find_spec(nombre_import) is not None
    version = "—"
    detalle = ""
    if disponible:
        try:
            modulo = importlib.import_module(nombre_import)
            modulos[nombre_import] = modulo
            version = getattr(modulo, "__version__", "instalado")
            estado = "OK"
        except Exception as exc:
            estado = "ERROR DE IMPORTACIÓN"
            detalle = f"{type(exc).__name__}: {exc}"
    else:
        estado = "FALTA"
        detalle = f"Instalar paquete: {nombre_paquete}"

    resultados_dependencias.append({
        "grupo": grupo,
        "paquete": nombre_paquete,
        "import": nombre_import,
        "versión": version,
        "estado": estado,
        "detalle": detalle,
    })

version_python_ok = sys.version_info[:2] == (3, 11)
resultados_dependencias.append({
    "grupo": "Entorno",
    "paquete": "python=3.11",
    "import": "sys",
    "versión": sys.version.split()[0],
    "estado": "OK" if version_python_ok else "ERROR",
    "detalle": "" if version_python_ok else "El entorno del taller requiere Python 3.11.",
})

if "pandas" in modulos and "IPython" in modulos:
    from IPython.display import display
    display(modulos["pandas"].DataFrame(resultados_dependencias))
else:
    for fila in resultados_dependencias:
        print(f"{fila['estado']:22s} | {fila['grupo']:14s} | {fila['paquete']} {fila['detalle']}")

faltantes = [fila for fila in resultados_dependencias if fila["estado"] != "OK"]
print(f"\nDependencias con problemas: {len(faltantes)} de {len(resultados_dependencias)}")
if faltantes:
    for fila in faltantes:
        print(f" - [{fila['grupo']}] {fila['paquete']}: {fila['estado']} {fila['detalle']}")

,grupo,paquete,import,versión,estado,detalle
0,Base,numpy,numpy,1.26.4,OK,
1,Base,pandas,pandas,2.2.3,OK,
2,Visualización,matplotlib,matplotlib,3.9.4,OK,
3,Geoespacial,geopandas,geopandas,0.14.4,OK,
4,Geoespacial,shapely,shapely,2.0.7,OK,
5,Geoespacial,rasterio,rasterio,1.4.3,OK,
6,Jupyter,ipython,IPython,9.13.0,OK,
7,Jupyter,ipywidgets,ipywidgets,8.1.9,OK,
8,Jupyter,ipykernel,ipykernel,7.2.0,OK,
9,Entorno,python=3.11,sys,3.11.17,OK,



Dependencias con problemas: 0 de 10


## 3. Estructura y datos del repositorio

La búsqueda identifica la raíz a partir de `environment.yml` y `data/`, tanto si Jupyter se inició desde la raíz como desde `notebooks/`. Solo se inspecciona `data/`, que contiene los insumos versionados del taller.

In [8]:
def encontrar_base_proyecto():
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "environment.yml").is_file() and (candidato / "data").is_dir():
            return candidato
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio (se requieren environment.yml y data/). "
        "Inicia Jupyter desde el repositorio o desde notebooks/."
    )


BASE_DIR = encontrar_base_proyecto()
DATA_DIR = BASE_DIR / "data"

carpetas_requeridas = ["data", "notebooks", "scripts"]
print("Raíz del proyecto:", BASE_DIR)
for nombre in carpetas_requeridas:
    ruta = BASE_DIR / nombre
    print(f"{'OK' if ruta.is_dir() else 'NO PRESENTE':12s} {nombre}/")

SHAPES_AU_DIR = DATA_DIR / "shapes_AU"
SHAPES_CYU_DIR = DATA_DIR / "shapes_CyU"
RASTERS_AU_DIR = DATA_DIR / "rasters_AU"
RASTERS_CYU_DIR = DATA_DIR / "rasters_CyU"

for nombre, ruta in [
    ("vectores Atlas Urbano", SHAPES_AU_DIR),
    ("vectores Cobertura y Uso", SHAPES_CYU_DIR),
    ("rasters Atlas Urbano", RASTERS_AU_DIR),
    ("rasters Cobertura y Uso", RASTERS_CYU_DIR),
]:
    print(f"{'OK' if ruta.is_dir() else 'NO PRESENTE':12s} {nombre}: {ruta.relative_to(BASE_DIR)}")

def buscar_archivos(carpeta, extensiones):
    if not carpeta.is_dir():
        return []
    extensiones = {extension.casefold() for extension in extensiones}
    return sorted(
        ruta for ruta in carpeta.rglob("*")
        if ruta.is_file() and ruta.suffix.casefold() in extensiones
    )


vectores_au = buscar_archivos(SHAPES_AU_DIR, {".shp", ".gpkg", ".geojson"})
vectores_cyu = buscar_archivos(SHAPES_CYU_DIR, {".shp", ".gpkg", ".geojson"})
rasters_au = buscar_archivos(RASTERS_AU_DIR, {".tif", ".tiff", ".vrt"})
rasters_cyu = buscar_archivos(RASTERS_CYU_DIR, {".tif", ".tiff", ".vrt"})

for etiqueta, archivos in [
    ("Vectores Atlas Urbano", vectores_au),
    ("Vectores Cobertura y Uso", vectores_cyu),
    ("Rasters Atlas Urbano", rasters_au),
    ("Rasters Cobertura y Uso", rasters_cyu),
]:
    print(f"{etiqueta}: {len(archivos)} archivo(s)")
    for archivo in archivos[:3]:
        print("  -", archivo.relative_to(BASE_DIR))
    if len(archivos) > 3:
        print(f"  ... y {len(archivos) - 3} más")

Raíz del proyecto: C:\Users\Paula_Copelli\Documents\COPERNICUS\TALLERES\02_IDE_URUGUAY_2026\repo\Taller-IDE-Uruguay
OK           data/
OK           notebooks/
OK           scripts/
OK           vectores Atlas Urbano: data\shapes_AU
OK           vectores Cobertura y Uso: data\shapes_CyU
OK           rasters Atlas Urbano: data\rasters_AU
OK           rasters Cobertura y Uso: data\rasters_CyU
Vectores Atlas Urbano: 1 archivo(s)
  - data\shapes_AU\municipios_20250507.shp
Vectores Cobertura y Uso: 3 archivo(s)
  - data\shapes_CyU\cuenca_laguna_merin\cuenca_laguna_merin.shp
  - data\shapes_CyU\limite_uruguay\limite_uruguay.shp
  - data\shapes_CyU\tiles_laguna_merin\tiles_laguna_merin.shp
Rasters Atlas Urbano: 4 archivo(s)
  - data\rasters_AU\2020\AU_UY_2020.tif
  - data\rasters_AU\2024\AU_UY_2024.tif
  - data\rasters_AU\homologacion_LCCS_UY\LCLU_UY_CLASE.tif
  ... y 1 más
Rasters Cobertura y Uso: 8 archivo(s)
  - data\rasters_CyU\nominal_coplac_2022_homologado.tif
  - data\rasters_CyU\nomina

## 4. Pruebas funcionales

Además de comprobar que los paquetes principales importan, esta sección prueba la lectura de las capas y rasters disponibles y la creación de widgets. No realiza transformaciones de coordenadas ni depende de paquetes que no se usan en los notebooks del taller.

In [9]:
pruebas = []

def registrar(servicio, prueba, estado, detalle=""):
    pruebas.append({
        "servicio": servicio,
        "prueba": prueba,
        "estado": estado,
        "detalle": detalle,
    })
    sufijo = f" — {detalle}" if detalle else ""
    print(f"{estado:7s} | {servicio:22s} | {prueba}{sufijo}")


# Lectura de capas vectoriales presentes en los datos versionados.
if "geopandas" in modulos:
    gpd = modulos["geopandas"]
    for servicio, archivos in [("Atlas Urbano", vectores_au), ("Cobertura y Uso", vectores_cyu)]:
        if not archivos:
            registrar(servicio, "Lectura de capa vectorial", "AVISO", "no hay una capa vectorial disponible en data/")
            continue
        try:
            capa = gpd.read_file(archivos[0])
            registrar(servicio, "Lectura de capa vectorial", "OK", f"{archivos[0].name}; {len(capa)} registros; CRS={capa.crs}")
        except Exception as exc:
            registrar(servicio, "Lectura de capa vectorial", "ERROR", f"{archivos[0].name}: {type(exc).__name__}: {exc}")
else:
    registrar("Datos vectoriales", "Lectura de capas", "OMITIDO", "geopandas ausente")


# Rasterio lee una ventana pequeña para limitar memoria.
if "rasterio" in modulos:
    rasterio = modulos["rasterio"]
    for servicio, archivos in [("Atlas Urbano", rasters_au), ("Cobertura y Uso", rasters_cyu)]:
        if not archivos:
            registrar(servicio, "Lectura de raster", "AVISO", "no hay un raster disponible en data/")
            continue
        try:
            with rasterio.open(archivos[0]) as src:
                ventana = src.read(1, window=((0, min(8, src.height)), (0, min(8, src.width))))
                detalle = f"{archivos[0].name}; {src.width}x{src.height}; CRS={src.crs}; dtype={ventana.dtype}"
            registrar(servicio, "Lectura de raster", "OK", detalle)
        except Exception as exc:
            registrar(servicio, "Lectura de raster", "ERROR", f"{archivos[0].name}: {type(exc).__name__}: {exc}")
else:
    registrar("Datos raster", "Lectura de rasters", "OMITIDO", "rasterio ausente")

if "ipywidgets" in modulos:
    try:
        _ = modulos["ipywidgets"].Dropdown(options=["Uruguay"], value="Uruguay", description="Test")
        registrar("Jupyter", "ipywidgets", "OK")
    except Exception as exc:
        registrar("Jupyter", "ipywidgets", "ERROR", f"{type(exc).__name__}: {exc}")
else:
    registrar("Jupyter", "ipywidgets", "OMITIDO", "ipywidgets ausente")

OK      | Atlas Urbano           | Lectura de capa vectorial — municipios_20250507.shp; 136 registros; CRS=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["EPSG","4326"]]
OK      | Cobertura y Uso        | Lectura de capa vectorial — cuenca_laguna_merin.shp; 1 registros; CRS=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AXIS["Longitude",EAST],AXIS["Latitude",NORTH]]
OK      | Atlas Urbano           | Lectura de raster — AU_UY_2020.tif; 11695x4738; CRS=GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH

## 5. Resultado

El entorno queda listo si todas las dependencias importan y ninguna prueba funcional produce **ERROR**. Los avisos sobre archivos opcionales no bloquean la validación.

In [10]:
problemas_dependencias = [fila for fila in resultados_dependencias if fila["estado"] != "OK"]
problemas_funcionales = [fila for fila in pruebas if fila["estado"] == "ERROR"]
avisos = [fila for fila in pruebas if fila["estado"] in ("AVISO", "OMITIDO")]

print("=" * 76)
print("RESUMEN — VERIFICACIÓN DEL ENTORNO IDE URUGUAY")
print("=" * 76)
print(f"Dependencias con problemas : {len(problemas_dependencias)}")
print(f"Pruebas funcionales ERROR  : {len(problemas_funcionales)}")
print(f"Avisos / pruebas omitidas   : {len(avisos)}")

if problemas_dependencias:
    print("\nDEPENDENCIAS A CORREGIR:")
    for fila in problemas_dependencias:
        print(f" - [{fila['grupo']}] {fila['paquete']}: {fila['estado']} {fila['detalle']}")

if problemas_funcionales:
    print("\nERRORES FUNCIONALES A CORREGIR:")
    for fila in problemas_funcionales:
        print(f" - [{fila['servicio']}] {fila['prueba']}: {fila['detalle']}")

if avisos:
    print("\nAVISOS (no bloquean por sí solos):")
    for fila in avisos:
        print(f" - [{fila['servicio']}] {fila['prueba']}: {fila['detalle']}")

if not problemas_dependencias and not problemas_funcionales:
    print("\nENTORNO LISTO PARA EL TALLER IDE URUGUAY.")
else:
    print("\nENTORNO NO LISTO. Corrige los problemas indicados antes de continuar.")

RESUMEN — VERIFICACIÓN DEL ENTORNO IDE URUGUAY
Dependencias con problemas : 0
Pruebas funcionales ERROR  : 0
Avisos / pruebas omitidas   : 0

ENTORNO LISTO PARA EL TALLER IDE URUGUAY.
